In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from package.RankAMIP.logistic import run_logistic_regression
from package.RankAMIP.data_script import make_BT_design_matrix
from package.RankAMIP.logistic import LogisticAMIP
from package.RankAMIP.logistic import find_closest_matchups
from package.RankAMIP.logistic import isRankingRobust
from package.RankAMIP.plot_util import *

### Is ChatBot Arena Data Robust?

### Load Data.

In [2]:
# Import datasets from https://huggingface.co/datasets/lmarena-ai/arena-human-preference-55k
from datasets import load_dataset
ds = load_dataset("lmarena-ai/arena-human-preference-55k")

/Users/JennyH/Library/Python/3.8/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# inspect the available splits
print(ds)  
# grab the ‘train’ split
train = ds["train"]

DatasetDict({
    train: Dataset({
        features: ['id', 'model_a', 'model_b', 'prompt', 'response_a', 'response_b', 'winner_model_a', 'winner_model_b', 'winner_tie'],
        num_rows: 57477
    })
})


In [4]:
df = train.to_pandas()
df.head()

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0


In [5]:
df.shape

(57477, 9)

In [6]:
# how to get the unique names in both columns
model_a_names = df['model_a'].unique()
model_b_names = df['model_b'].unique()
# combine the two arrays and get the unique names
model_names = np.unique(np.concatenate((model_a_names, model_b_names)))
# print the number of unique model names
print(f"Number of unique model names: {len(model_names)}")

Number of unique model names: 64


In [56]:
ties = df[df['winner_tie'] == 1]
print(f"Number of ties: {len(ties)}")
# proportion of ties.
print(f"Proportion of ties: {len(ties) / len(df):.2%}")

Number of ties: 17761
Proportion of ties: 30.90%


#### Keep ties.

In [7]:
# drop rows in df with df['winner_tie'] == 1
rawBT = df[['model_a', 'model_b', 'winner_model_a', 'winner_tie']]
rawBT.head()

,model_a,model_b,winner_model_a,winner_tie
0,gpt-4-1106-preview,gpt-4-0613,1,0
1,koala-13b,gpt-4-0613,0,0
2,gpt-3.5-turbo-0613,mistral-medium,0,1
3,llama-2-13b-chat,mistral-7b-instruct,1,0
4,koala-13b,gpt-3.5-turbo-0314,0,0


In [ ]:
# to test robustness upon filtering out these models (do not run this cell).
# rawBT = rawBT[~((rawBT['model_a'].isin(['gpt-4-1106-preview', 'gpt-4-0314', 'gpt-4-0613', 'qwen1.5-72b-chat', 'mistral-medium', 'claude-1', 'claude-2.0', 'gemini-pro-dev-api', 'yi-34b-chat'])) | 
#                 (rawBT['model_b'].isin(['gpt-4-1106-preview', 'gpt-4-0314', 'gpt-4-0613', 'qwen1.5-72b-chat', 'mistral-medium', 'claude-1', 'claude-2.0', 'gemini-pro-dev-api', 'yi-34b-chat'])))]

In [8]:
for model in model_names:
    filtered = rawBT[
        (rawBT['model_a'] == model) | 
        (rawBT['model_b'] == model)
    ]
    print(f"{model}: {filtered.shape[0]}")

RWKV-4-Raven-14B: 1158
alpaca-13b: 1403
chatglm-6b: 1261
chatglm2-6b: 564
chatglm3-6b: 989
claude-1: 3978
claude-2.0: 2456
claude-2.1: 5583
claude-instant-1: 4136
codellama-34b-instruct: 1474
deepseek-llm-67b-chat: 795
dolly-v2-12b: 800
dolphin-2.2.1-mistral-7b: 373
falcon-180b-chat: 286
fastchat-t5-3b: 1021
gemini-pro: 1438
gemini-pro-dev-api: 1486
gpt-3.5-turbo-0125: 861
gpt-3.5-turbo-0314: 1302
gpt-3.5-turbo-0613: 7083
gpt-3.5-turbo-1106: 3352
gpt-4-0125-preview: 1160
gpt-4-0314: 4122
gpt-4-0613: 6165
gpt-4-1106-preview: 7387
gpt4all-13b-snoozy: 408
guanaco-33b: 684
koala-13b: 1598
llama-13b: 547
llama-2-13b-chat: 2607
llama-2-70b-chat: 3428
llama-2-7b-chat: 1793
llama2-70b-steerlm-chat: 667
mistral-7b-instruct: 1617
mistral-7b-instruct-v0.2: 100
mistral-medium: 3315
mixtral-8x7b-instruct-v0.1: 3545
mpt-30b-chat: 598
mpt-7b-chat: 928
nous-hermes-2-mixtral-8x7b-dpo: 325
oasst-pythia-12b: 1494
openchat-3.5: 1632
openchat-3.5-0106: 244
openhermes-2.5-mistral-7b: 952
palm-2: 1977
pplx-7

In [9]:
# make weighted design matrix for BT.
X, y, player_to_id = make_BT_design_matrix(rawBT, weight_tie = True)
X.shape, y.shape

((114954, 63), (114954,))

In [ ]:
# import pickle
# with open('bt_data.pkl', 'wb') as f:
#     pickle.dump((X, y), f)


In [ ]:
# # Results upon removing players 2 through 10 in the dataset.
# with open('results/ChatbotArenaNonrobustWtd.pkl', 'rb') as f:
#     cba_results = pickle.load(f)
# cba_results

{(1, 2): (21,
  None,
  -0.004146461884523535,
  0.002200252892290602,
  array([46259,  6212])),
 (3, 29): (6,
  61,
  0.7214777505716051,
  -0.026849342132582388,
  array([26551, 17137, 40808, 32585,  3046, 11617, 39409, 39779, 33011,
         56011, 24859, 28859, 48893, 17324, 38734, 13179, 18149, 34968,
         37289, 20538,  1308,  5210, 23104, 49522,  7020, 46358, 27533,
         48536, 40704])),
 (5, 3): (41,
  47,
  0.01376073850843107,
  -0.001148770551445799,
  array([20425, 38755, 13835])),
 (10, 1): (5,
  4,
  0.0008246616768733395,
  -0.0012153237089130853,
  array([24811])),
 (20, 1): (35,
  48,
  0.007468269893045054,
  -0.0006298890499366605,
  array([5389]))}

In [10]:
id_to_player = {v: k for k, v in player_to_id.items()}
id_to_player

{0: 'gpt-4-1106-preview',
 1: 'koala-13b',
 2: 'gpt-3.5-turbo-0613',
 3: 'llama-2-13b-chat',
 4: 'vicuna-13b',
 5: 'mixtral-8x7b-instruct-v0.1',
 6: 'gemini-pro',
 7: 'gpt-4-0314',
 8: 'vicuna-7b',
 9: 'chatglm3-6b',
 10: 'pplx-70b-online',
 11: 'mpt-30b-chat',
 12: 'llama2-70b-steerlm-chat',
 13: 'claude-1',
 14: 'claude-2.1',
 15: 'chatglm-6b',
 16: 'claude-instant-1',
 17: 'dolly-v2-12b',
 18: 'claude-2.0',
 19: 'deepseek-llm-67b-chat',
 20: 'openchat-3.5',
 21: 'starling-lm-7b-alpha',
 22: 'gpt-4-0125-preview',
 23: 'llama-2-7b-chat',
 24: 'gpt-4-0613',
 25: 'wizardlm-70b',
 26: 'stablelm-tuned-alpha-7b',
 27: 'vicuna-33b',
 28: 'chatglm2-6b',
 29: 'dolphin-2.2.1-mistral-7b',
 30: 'llama-2-70b-chat',
 31: 'llama-13b',
 32: 'palm-2',
 33: 'wizardlm-13b',
 34: 'codellama-34b-instruct',
 35: 'gemini-pro-dev-api',
 36: 'gpt-3.5-turbo-0314',
 37: 'gpt-3.5-turbo-1106',
 38: 'yi-34b-chat',
 39: 'oasst-pythia-12b',
 40: 'qwen-14b-chat',
 41: 'alpaca-13b',
 42: 'qwen1.5-72b-chat',
 43: 'gpt

#### Run Top-k Robustness Check.

In [11]:
ks = [1] # 2
results = {}
for k in ks:
    alphaN = 1
    chatbotA = -1
    while chatbotA == -1:
        chatbotA, chatbotB, chatbotOriginalBetaDiff, chatNewBetaDiff, chatIndices = isRankingRobust(k, alphaN, X, y, weighted = True)
        results[(k, alphaN)] = (chatbotA, chatbotB, chatbotOriginalBetaDiff, chatNewBetaDiff, chatIndices)
        alphaN += 1
        print(alphaN)

2
3


Plot arena BT-scores.

In [13]:
rankings = return_rankings_list(X, y, results, 1, 2, player_to_id)

In [12]:
results_nonrobust = {k: v for k, v in results.items() if v[0] != -1}
results_nonrobust

{(1, 2): (21,
  None,
  -0.004146461884523535,
  0.002200252892290602,
  array([46259,  6212]))}

In [51]:
def plot_bt_scores(X, y, rankings, alphaN, topk, plot_title, filename_to_save):
    """
    Plots BT scores before and after data removal.
    
    Args:
    X: np.ndarray, the design matrix.
    y: np.ndarray, the response variable.
    rankings: list of tuples, (model_name, full_score, old_score, new_score)
    alphaN: int, number of dropped matches
    topk: int, number of top models to display
    plot_title: str, title of the plot
    filename_to_save: str, path to save the figure
    """
    # Extract top-k entries
    # Sorted by old_scores (index 2) in descending.
    rankings = sorted(rankings[:topk], key=lambda x: x[2], reverse=False)
    model_names = [x[0] for x in rankings[:topk]]
    old_scores = [x[2] for x in rankings[:topk]]
    new_scores = [x[3] for x in rankings[:topk]]
    num_matches_total = len(y)
    y_plot = np.arange(len(rankings[:topk]))
    offset = 0.15
    # Plot.
    # Set global font to monospace and increase default font size
    plt.rcParams.update({
        'font.family': 'monospace',
        'font.size': 14
    })

    # Plot
    plt.figure(figsize=(13, 6), dpi=250)

    # Remove top and right spines
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Scatter
    plt.scatter(old_scores, y_plot, marker='o', color='#fb7d4d', s=96) # label='BT Score Full Data',
    # plt.scatter(new_scores, y_plot + offset, 
    #             label=f'BT Score After Dropping {alphaN} out of {num_matches_total}\n matches ({(alphaN/num_matches_total * 100):.3f}%)',
    #             marker='s', color='orange', s=72)

    # Extend x-axis limits slightly to the left and right
    min_score = min(old_scores)
    max_score = max(old_scores) # max(max(old_scores), max(new_scores))
    plt.xlim(min_score - 0.05, max_score + 0.05)

    # Annotate scores next to points
    for i in range(len(y_plot)):
        if i > len(y_plot) - 3:
            # Position the text to the left of the point
            plt.text(old_scores[i] - 0.03, y_plot[i], f'{old_scores[i]:.3f}', 
                    va='center', ha='right', fontsize=17, fontfamily='monospace', color='#e97442')
        else:
            # Position the text to the right of the point
            plt.text(old_scores[i] + 0.03, y_plot[i], f'{old_scores[i]:.3f}', 
                    va='center', ha='left', fontsize=17, fontfamily='monospace', color='#e97442')

    # Axis
    plt.xlabel('Bradley-Terry Score', fontsize=22, fontfamily='monospace')
    plt.yticks(y_plot, model_names, fontsize=19, fontfamily='monospace')
    plt.xticks(fontsize=19, fontfamily='monospace')
    # plt.title(plot_title, fontsize=22, fontfamily='monospace')
    # plt.legend(fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()

    # Save
    plt.savefig(filename_to_save, bbox_inches='tight')
    plt.close()

In [52]:
# plot top-20 models on full chatbot arena.
filename_to_save = 'fig/top10_cba.png'
plot_title = 'Model Rankings in Chatbot Arena'
plot_bt_scores(X, y, rankings, 1, 10, plot_title, filename_to_save)

In [20]:
ks = [5, 10, 20]
results2 = {}
for k in ks:
    alphaN = 1
    chatbotA = -1
    while chatbotA == -1:
        chatbotA, chatbotB, chatbotOriginalBetaDiff, chatNewBetaDiff, chatIndices = isRankingRobust(k, alphaN, X, y, weighted = True)
        results2[(k, alphaN)] = (chatbotA, chatbotB, chatbotOriginalBetaDiff, chatNewBetaDiff, chatIndices)
        alphaN += 1

In [ ]:
results_nonrobust_2 = {k: v for k, v in results2.items() if v[0] != -1}
results_nonrobust_2

In [ ]:
# combine results_nonrobust_1 and results_nonrobust_2.
results_nonrobust = {**results_nonrobust_1, **results_nonrobust_2}
results_nonrobust
# save results as a .pkl file.
# import pickle
# with open('results/ChatbotArenaNonrobustWtd.pkl', 'wb') as f:
#     pickle.dump(results_nonrobust, f)

In [45]:
results_nonrobust

{(1, 2): (21,
  -1,
  -0.004146461884523535,
  0.002200252892290602,
  array([46259,  6212])),
 (3, 29): (6,
  61,
  0.7214777505716051,
  -0.026849342132582388,
  array([26551, 17137, 40808, 32585,  3046, 11617, 39409, 39779, 33011,
         56011, 24859, 28859, 48893, 17324, 38734, 13179, 18149, 34968,
         37289, 20538,  1308,  5210, 23104, 49522,  7020, 46358, 27533,
         48536, 40704])),
 (5, 3): (41,
  47,
  0.01376073850843107,
  -0.001148770551445799,
  array([20425, 38755, 13835])),
 (10, 1): (5,
  4,
  0.0008246616768733395,
  -0.0012153237089130853,
  array([24811])),
 (20, 1): (35,
  48,
  0.007468269893045054,
  -0.0006298890499366605,
  array([5389]))}

#### Below, we inspect the ranking flip between the first- and second-place models.

In [39]:
### Load in results.
import pickle
with open("results/ChatbotArenaNonrobustWtd.pkl", "rb") as f:
    chatBotArenaDataDropped = pickle.load(f)

In [40]:
# chatBotArena_noTies = pd.read_csv("data/chatBotArena_noTies.csv")
# chatBotArena_noTies.head()
df.head()

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie,model_pair
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0,"(gpt-4-0613, gpt-4-1106-preview)"
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0,"(gpt-4-0613, koala-13b)"
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1,"(gpt-3.5-turbo-0613, mistral-medium)"
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0,"(llama-2-13b-chat, mistral-7b-instruct)"
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0,"(gpt-3.5-turbo-0314, koala-13b)"


9 evals were dropped to flip model i.d. 16 and i.d. None (the reference model).

The name of these models are: 
('gpt-4-0125-preview', 0: 'gpt-4-1106-preview')

In [31]:
## Count number of games between the two models that changed ranks.
is_gpt41106_gpt40125 = (
    (df['model_a'].str.contains('gpt-4-0125-preview') & df['model_b'].str.contains('gpt-4-1106-preview')) |
    (df['model_a'].str.contains('gpt-4-1106-preview') & df['model_b'].str.contains('gpt-4-0125-preview'))
)

num_gpt41106_gpt40125 = df[is_gpt41106_gpt40125].shape[0]
print("Number of games between GPT-4-1106 and GPT-4-0125: ", num_gpt41106_gpt40125)

Number of games between GPT-4-1106 and GPT-4-0125:  134


In [11]:
# model pairs (sorted to group symmetric pairs)
df['model_pair'] = df.apply(lambda row: tuple(sorted([row['model_a'], row['model_b']])), axis=1)
df['model_pair']

0                (gpt-4-0613, gpt-4-1106-preview)
1                         (gpt-4-0613, koala-13b)
2            (gpt-3.5-turbo-0613, mistral-medium)
3         (llama-2-13b-chat, mistral-7b-instruct)
4                 (gpt-3.5-turbo-0314, koala-13b)
                           ...                   
57472                      (claude-1, gpt-4-0613)
57473              (claude-2.0, llama-2-13b-chat)
57474                      (alpaca-13b, claude-1)
57475                    (palm-2, tulu-2-dpo-70b)
57476    (gemini-pro-dev-api, gpt-4-1106-preview)
Name: model_pair, Length: 57477, dtype: object

In [12]:
# Count number of games per model pair
pair_counts = df['model_pair'].value_counts()

# Compute average
average_games_per_pair = pair_counts.mean()
print("Average number of games per model pair:", average_games_per_pair) # 31.90

Average number of games per model pair: 45.08


In [ ]:
# Find the win margin between 'gpt-4-0125-preview' and 'gpt-4-1106-preview'
# that is, find all games that are between the two models.
dfFlippedRanking = df[is_gpt41106_gpt40125]
## Count number of games between that 'gpt-4-0125-preview' won.
gpt40125_wins = (
    (dfFlippedRanking['model_a'].str.contains('gpt-4-0125-preview') & dfFlippedRanking['winner_model_a'] == 1) |
    (dfFlippedRanking['model_b'].str.contains('gpt-4-0125-preview') & dfFlippedRanking['winner_model_b'] == 1)
)
num_gpt40125_wins = dfFlippedRanking[gpt40125_wins].shape[0]
 # 0.5373134328358209

Proportion of games that GPT-4-0125 won:  0.26865671641791045


In [14]:
gpt41106_wins = (
    (dfFlippedRanking['model_a'].str.contains('gpt-4-1106-preview') & dfFlippedRanking['winner_model_a'] == 1) |
    (dfFlippedRanking['model_b'].str.contains('gpt-4-1106-preview') & dfFlippedRanking['winner_model_b'] == 1)
)

num_gpt41106_wins = dfFlippedRanking[gpt41106_wins].shape[0]

In [ ]:
print("Proportion of games that GPT-4-0125 won: ", num_gpt40125_wins / (num_gpt40125_wins + num_gpt41106_wins))

Proportion of games that GPT-4-0125 won:  0.5373134328358209


In [27]:
# dropping 3 points flips the 5th and 6th place models.
id_to_player[48], id_to_player[42]

('mistral-medium', 'qwen1.5-72b-chat')

#### Player involvement in dropped matches

In [77]:
chatBotArenaDataDropped

{(1, 2): (21,
  -1,
  -0.004146461884523535,
  0.002200252892290602,
  array([46259,  6212])),
 (3, 29): (6,
  61,
  0.7214777505716051,
  -0.026849342132582388,
  array([26551, 17137, 40808, 32585,  3046, 11617, 39409, 39779, 33011,
         56011, 24859, 28859, 48893, 17324, 38734, 13179, 18149, 34968,
         37289, 20538,  1308,  5210, 23104, 49522,  7020, 46358, 27533,
         48536, 40704])),
 (5, 3): (41,
  47,
  0.01376073850843107,
  -0.001148770551445799,
  array([20425, 38755, 13835])),
 (10, 1): (5,
  4,
  0.0008246616768733395,
  -0.0012153237089130853,
  array([24811])),
 (20, 1): (35,
  48,
  0.007468269893045054,
  -0.0006298890499366605,
  array([5389]))}

In [34]:
with open('/Users/JennyH/Desktop/IsRankingRobust/results/ChatbotArenaNonrobustWtd.pkl', 'rb') as f:
    chatBotArenaDataDropped = pickle.load(f)

### Manuel check (for each k) on the MIS! 

In [35]:
results_nonrobust = chatBotArenaDataDropped

In [36]:
old_tuple = chatBotArenaDataDropped[(1, 2)] # = 0 # change none to -1 (gpt-4-1106-preview).
new_tuple = tuple(-1 if i == 1 and val is None else val for i, val in enumerate(old_tuple))
results_nonrobust[(1, 2)] = new_tuple

In [44]:
rows = []
for (k, aN), (playerA, playerB, original_beta_diff, new_beta_diff_refit, indices) in results_nonrobust.items():
    rows.append({
        "k-aN": (k, aN),
        "playerA": playerA + 1, # to account for the reference index.
        "playerB": playerB + 1,
        "original_beta_diff": original_beta_diff,
        "new_beta_diff_refit": new_beta_diff_refit,
        "indices": indices
    })
cba_results = pd.DataFrame(rows)
cba_results.head()

,k-aN,playerA,playerB,original_beta_diff,new_beta_diff_refit,indices
0,"(1, 2)",22,0,-0.004146,0.002200,"[46259, 6212]"
1,"(3, 29)",7,62,0.721478,-0.026849,"[26551, 17137, 40808, 32585, 3046, 11617, 3940..."
2,"(5, 3)",42,48,0.013761,-0.001149,"[20425, 38755, 13835]"
3,"(10, 1)",6,5,0.000825,-0.001215,[24811]
4,"(20, 1)",36,49,0.007468,-0.000630,[5389]


In [38]:
cba_results['indices'][0]

array([46259,  6212])

In [39]:
# reverse the mapping.
id_to_player = {v: k for k, v in player_to_id.items()}

In [40]:
id_to_player[0], id_to_player[22]

('llama-2-13b-chat', 'llama-13b')

In [47]:
# read in the results.
cba_results.head()
cba_results['playerA_Name'] = cba_results['playerA'].map(id_to_player)
cba_results['playerB_Name'] = cba_results['playerB'].map(id_to_player)

In [48]:
cba_results.head()

,k-aN,playerA,playerB,original_beta_diff,new_beta_diff_refit,indices,playerA_Name,playerB_Name
0,"(1, 2)",22,0,-0.004146,0.002200,"[46259, 6212]",gpt-4-0125-preview,gpt-4-1106-preview
1,"(3, 29)",7,62,0.721478,-0.026849,"[26551, 17137, 40808, 32585, 3046, 11617, 3940...",gpt-4-0314,mistral-7b-instruct-v0.2
2,"(5, 3)",42,48,0.013761,-0.001149,"[20425, 38755, 13835]",qwen1.5-72b-chat,mistral-medium
3,"(10, 1)",6,5,0.000825,-0.001215,[24811],gemini-pro,mixtral-8x7b-instruct-v0.1
4,"(20, 1)",36,49,0.007468,-0.000630,[5389],gpt-3.5-turbo-0314,nous-hermes-2-mixtral-8x7b-dpo


In [76]:
# for each index, find the corresponding row in the original dataframe.
indices = cba_results['indices'][0] 
rawBT.iloc[indices]
# find the proportion of games where neither 

,model_a,model_b,winner_model_a,winner_tie
46259,vicuna-13b,gpt-4-0125-preview,1,0
6212,stripedhyena-nous-7b,gpt-4-0125-preview,1,0


In [ ]:
# k=1. 2 games were dropped to flip models gpt-4-0125-preview (1st place) with gpt-4-1106-preview (2nd place). 
# all dropped matches were between gpt-4-0125-preview and vicuna-13b, and stripedhyena-nous-7b, with gpt-4-0125-preview winning.


# k=3. 29 games were dropped to flip models gpt-4-0314 (3rd place) with mistral-7b-instruct-v0.2 (6th place).
# all games were between mistral-7b-instruct-v0.2 and various other models, mistral-7b-instruct-v0.2 loses all of these matches.


# k=5. 3 games were dropped to flip models qwen1.5-72b-chat (5th place) with mistral-medium (6th place).
# all dropped matches were between qwen1.5-72b-chat and  gpt-4-1106-preview (1st place), with qwen1.5-72b-chat (5th place) winning.

# k=10. 1 game was dropped to flip models gemini-pro (10th) and mixtral-8x7b-instruct-v0.1 (11th place).
# the dropped match was between the two models, with gemini-pro winning.

# k=20. 1 games were dropped to flip models gpt-3.5-turbo-0314 (20th place) and nous-hermes-2-mixtral-8x7b-dpo (21st place).
# the dropped match is between nous-hermes-2-mixtral-8x7b-dpo (21st place) and vicuna-13b (22st place), with nous-hermes-2-mixtral-8x7b-dpo losing.


In [ ]:
# k=1. 9 games were dropped to flip models gpt-4-0125-preview (1st place) with gpt-4-1106-preview (2nd place). 
# all dropped matches were between these two models, with gpt-4-0125-preview winning.
# k=3. 24 games were dropped to flip models gpt-4-0314(2nd place) with qwen1.5-72b-chat (5th place).
# all games were between qwen1.5-72b-chat and various other models, qwen1.5-72b-chat loses all of these matches.
# k=5. 5 games were dropped to flip models qwen1.5-72b-chat (5th place) with mistral-medium (6th place).
# all dropped matches were between gpt-4-1106-preview (2nd place) and qwen1.5-72b-chat, with qwen1.5-72b-chat (5th place) winning.
# k=10. 3 games were dropped to flip models yi-34b-chat (10th) and gemini-pro (11th place).
# all dropped matches are between yi-34b-chat and gemini-pro, with yi-34b-chat winning.
# k=20. 2 games were dropped to flip models nous-hermes-2-mixtral-8x7b-dpo (20th place) and 'vicuna-33b'(21st place).
# all dropped matches are between gpt-4-1106-preview (2nd place) and nous-hermes-2-mixtral-8x7b-dpo, with nous-hermes-2-mixtral-8x7b-dpo winning.

# (namely, vicuna-33b, qwen1.5-7b-chat, tulu-2-dpo-70b, gpt-3.5-turbo-1106, llama-2-13b-chat, )

### Inspect the dropped human evals.

In [174]:
chatbot_arena_results_with_props = add_match_proportions(cba_results, chatBotArena_noTies)
chatbot_arena_results_with_props.head()

,k-aN,playerA,playerB,original_beta_diff,new_beta_diff_refit,indices,playerA_Name,playerB_Name,prop_both,prop_one,prop_neither
0,"(1, 9)",17,0,0.033658,-0.001239,"[2370, 3227, 38242, 10155, 11353, 17714, 18638...",gpt-4-0125-preview,gpt-4-1106-preview,1.0,0.0,0.0
1,"(3, 24)",7,31,0.213213,-0.007985,"[24342, 29361, 38576, 36203, 12188, 15215, 364...",gpt-4-0314,qwen1.5-72b-chat,0.0,1.0,0.0
2,"(5, 5)",31,39,0.036573,-0.006157,"[9754, 19106, 20511, 9565, 26870]",qwen1.5-72b-chat,mistral-medium,0.0,1.0,0.0
3,"(10, 3)",40,5,0.011066,-0.002069,"[16514, 601, 7082]",yi-34b-chat,gemini-pro,1.0,0.0,0.0
4,"(20, 2)",57,20,0.015632,-0.013790,"[2783, 22323]",nous-hermes-2-mixtral-8x7b-dpo,vicuna-33b,0.0,1.0,0.0


In the rank-flipping between mistral and qwen, a total of 5 evaluations where qwen played and won against gpt-4-1106-preview was enough to move its ranking to above that of mistral.

In [ ]:
# Let's investigate the responses that changed the Top-5 models.
mistral_qwen_inds = chatbot_arena_results_with_props["indices"].iloc[2]
chatBotArena_noTies.iloc[mistral_qwen_inds]

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
9754,1051544564,qwen1.5-72b-chat,gpt-4-1106-preview,"[""\nread this content. Do not change the conte...","[""6. Use AI Writing Tools\n\nIncorporating AI ...","[""6. Use AI to Your Advantage\n\nIn this age o...",1,0,0
19106,2049273519,gpt-4-1106-preview,qwen1.5-72b-chat,"[""Which is the newest model that also contain ...","[""I'm sorry, but your question isn't clear. Wh...","[""As an AI, I don't have real-time access to u...",0,1,0
20511,2199125192,qwen1.5-72b-chat,gpt-4-1106-preview,"[""Please answer this questions in 8 sentences ...","[""1. When a legal description appears clear in...","[""1. The type of ambiguity encountered is like...",1,0,0
9565,1029897539,gpt-4-1106-preview,qwen1.5-72b-chat,"[""Is it possible to give a transformer custom ...","[""Yes, it is possible to give a transformer mo...","[""Yes, it is possible to give a transformer mo...",0,1,0
26870,2888250053,gpt-4-1106-preview,qwen1.5-72b-chat,"[""How to use Poetry to install packages in pyt...","[""Poetry is a tool for dependency management a...","[""Poetry is a dependency manager for Python th...",0,1,0


In [261]:
player_to_id

{'gpt-4-1106-preview': 0,
 'koala-13b': 1,
 'llama-2-13b-chat': 2,
 'vicuna-13b': 3,
 'mixtral-8x7b-instruct-v0.1': 4,
 'gemini-pro': 5,
 'gpt-3.5-turbo-0613': 6,
 'gpt-4-0314': 7,
 'vicuna-7b': 8,
 'chatglm3-6b': 9,
 'pplx-70b-online': 10,
 'mpt-30b-chat': 11,
 'llama2-70b-steerlm-chat': 12,
 'claude-1': 13,
 'chatglm-6b': 14,
 'claude-2.0': 15,
 'starling-lm-7b-alpha': 16,
 'gpt-4-0125-preview': 17,
 'llama-2-7b-chat': 18,
 'stablelm-tuned-alpha-7b': 19,
 'vicuna-33b': 20,
 'gpt-4-0613': 21,
 'dolphin-2.2.1-mistral-7b': 22,
 'palm-2': 23,
 'wizardlm-13b': 24,
 'claude-2.1': 25,
 'claude-instant-1': 26,
 'gpt-3.5-turbo-1106': 27,
 'oasst-pythia-12b': 28,
 'qwen-14b-chat': 29,
 'openchat-3.5': 30,
 'qwen1.5-72b-chat': 31,
 'codellama-34b-instruct': 32,
 'deepseek-llm-67b-chat': 33,
 'gpt-3.5-turbo-0125': 34,
 'pplx-7b-online': 35,
 'qwen1.5-4b-chat': 36,
 'fastchat-t5-3b': 37,
 'llama-2-70b-chat': 38,
 'mistral-medium': 39,
 'yi-34b-chat': 40,
 'zephyr-7b-beta': 41,
 'openhermes-2.5-mi

In [194]:
# MIS: prompts.
# for prompt in chatBotArena_noTies.iloc[gpt4top2_inds]['prompt']:
#     print(prompt)
import textwrap
for i, prompt in enumerate(chatBotArena_noTies.loc[mistral_qwen_inds]['prompt']):
    print(f"\n=== prompt {i+1} ===\n")
    print(textwrap.fill(prompt, width=100))


=== prompt 1 ===

["\nread this content. Do not change the content written. Add 5 more tips in the same tone of voice
and style. \nhave one be use AI \n\nHey freelance writers: Let's talk about writer's block.\n\nI
know, I know. You don't want to say it out loud. It's like \"Voldemort\" or actors saying
\"Macbeth\" out loud (that's why they call it \"The Scottish Play\"). But, like the common cold,
writer's block is out there ... and odds are, it's going to get every freelance writer at some
point.\n\nThe Terror of Writer's Block\n\nYou've probably had it happen to you at one time or
another. You're sitting at your desk, a cup of coffee or cold beverage nearby, ready to start
writing, knocking off those assignments, and adding some zeroes to the bank account. The words are
ready to go ... but somehow, someway, they just won't come out. The blank page remains there,
looking at you, taunting you.\n\nThen the terror starts. \"Oh no,\" you think. \"How long will this
last? Will I get para

In [195]:
# MIS: Answer ModelA.
# for response_A in chatBotArena_noTies.iloc[gpt4top2_inds]['response_a']:
#     print(response_A)
import textwrap

for i, response_a in enumerate(chatBotArena_noTies.loc[mistral_qwen_inds]['response_a']):
    print(f"\n=== Response {i+1} ===\n")
    print(textwrap.fill(response_a, width=100))


=== Response 1 ===

["6. Use AI Writing Tools\n\nIncorporating AI writing tools can occasionally help break the creative
gridlock. There are various AI assistants that can generate prompts, suggest sentences, or even
complete paragraphs for you. Of course, you'll want to maintain your unique voice and edit AI-
generated content, but it's a useful starting point to overcome writer's block. It can spark new
ideas or simply provide a gentle nudge to get your creative juices flowing.\n\n7. Engage in Physical
Exercise\n\nPhysical activity can help clear your mind and boost your energy levels. Go for a run,
hit the gym, or do a brief workout session. Exercise releases endorphins that can improve your mood
and stimulate your brain, making it easier to overcome writer's block. Plus, you'll feel refreshed
and ready to tackle your writing once you're done.\n\n8. Set Realistic Writing Goals\n\nSometimes,
the pressure of a looming deadline or an overly ambitious goal can contribute to writer's bl

In [196]:
import textwrap

for i, response_b in enumerate(chatBotArena_noTies.loc[mistral_qwen_inds]['response_b']):
    print(f"\n=== Response {i+1} ===\n")
    print(textwrap.fill(response_b, width=100))


=== Response 1 ===

["6. Use AI to Your Advantage\n\nIn this age of technology, leverage the power of AI to overcome
writer's block. AI writing tools can provide prompts, continue a thought, or even help outline your
piece. It's like having a brainstorming buddy at your fingertips, one who's available 24\/7 and
doesn't mind you bouncing ideas off of them, no matter how rough they might be. Just type in a
sentence or a theme, and let the artificial intelligence give you a nudge in the right direction.
Remember, it's not about replacing your creativity but enhancing it.\n\n7. Set Small, Achievable
Goals\n\nSometimes writer's block stems from feeling overwhelmed by the size of the project. Break
it down into small, manageable pieces. Set a goal for a paragraph, a page, or maybe just a great
opening sentence. Achieving these smaller goals can give you a sense of progress and help build
momentum. As you tick off these mini milestones, the larger project won't seem so daunting.\n\n8.
Engage